
# TDA Conference Demo — Persistent Homology for Link Prediction

**박준영** · TDA 학회 · 2026-06-21
GitHub: github.com/jjune5/TDA_conference

## Outline
1. 데이터 로드 + 그래프 통계
2. Persistent Diagram 계산 (한 edge에 대해)
3. Persistence Image 변환
4. 3-way 비교 결과 (TLC-GNN / PDGNN / No PI)
5. SBM density × heterophily sweep 결과
6. Adaptive PI Gating 결과


In [ ]:

import sys, os
sys.path.insert(0, '..')  # for loaddatas, Knowledge_Distillation, etc.
import numpy as np
import torch
import networkx as nx
import matplotlib.pyplot as plt
import loaddatas as lds

# Load Cora (small, easy demo dataset)
ds = lds.loaddatas('Cora')
data = ds[0]
print(f'Cora: {data.num_nodes} nodes, {data.edge_index.size(1)//2} edges')
print(f'Feature dim: {data.x.size(1)}, num classes: {ds.num_classes}')



## Persistent Diagram 계산

각 edge (u, v)에 대해:
1. u, v의 1-hop 이웃 교집합 (vicinity subgraph) 추출
2. Ollivier-Ricci curvature 기반 filter 적용
3. dionysus로 extended persistence diagram 계산


In [ ]:

from Knowledge_Distillation.prepare_data_LP_modern import _edge_vicinity, _ollivier_ricci_filt
from Knowledge_Distillation.accelerated_PD import perturb_filter_function, Union_find, Accelerate_PD

# Build full graph + Ricci
g = nx.Graph()
g.add_nodes_from(range(data.num_nodes))
ei = np.array(data.edge_index)
g.add_edges_from((int(ei[0, i]), int(ei[1, i])) for i in range(ei.shape[1]))
ricci_list = lds.compute_ricci_curvature(data)
ricci_lookup = {(int(a), int(b)): float(c) for a, b, c in ricci_list}
for a, b in g.edges():
    w = ricci_lookup.get((a, b), ricci_lookup.get((b, a), 0.0)) + 1
    g[a][b]['weight'] = max(w, 1e-6)

# Pick one edge with non-trivial vicinity
example_edges = [(int(ei[0, i]), int(ei[1, i])) for i in range(min(100, ei.shape[1]))]
for u, v in example_edges:
    sub = _edge_vicinity(g, u, v, hop=2)
    if sub.number_of_nodes() >= 5:
        break
print(f'Example edge: ({u}, {v}), vicinity: {sub.number_of_nodes()} nodes, {sub.number_of_edges()} edges')

# Compute PD
filt_vals = _ollivier_ricci_filt(sub, u, v, ricci_lookup)
sub_re = nx.convert_node_labels_to_integers(sub)
sf = perturb_filter_function(sub_re, filt_vals)
PD_up, ess0, PD_down, Pos, Neg = Union_find(sf)
PD_one = Accelerate_PD(Pos, Neg, sf)

# Combine all PD points
pd_all = []
for arr in [PD_up, ess0, PD_down, PD_one]:
    a = np.asarray(arr, dtype=np.float64).reshape(-1, 2) if len(arr) else np.empty((0, 2))
    if a.size:
        pd_all.append(a)
pd_all = np.concatenate(pd_all, axis=0) if pd_all else np.empty((0, 2))
print(f'PD shape: {pd_all.shape} (rows = persistence pairs)')

# Plot PD
fig, ax = plt.subplots(figsize=(6, 6))
if pd_all.size:
    ax.scatter(pd_all[:, 0], pd_all[:, 1], s=60, alpha=0.7)
    lo, hi = pd_all.min(), pd_all.max()
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.3, label='diagonal')
ax.set_xlabel('birth'); ax.set_ylabel('death')
ax.set_title(f'Persistent Diagram for edge ({u}, {v}) in Cora')
ax.legend()
plt.tight_layout()
plt.show()



## Persistence Image (PI)

PD를 5×5 격자에 Gaussian smoothing → 25-dim vector. ML 입력으로 사용 가능.


In [ ]:

from sg2dgm import PersistenceImager as pimg_mod
imager = pimg_mod.PersistenceImager(resolution=5)
PI = imager.transform(pd_all) if pd_all.size else np.zeros((5, 5))
PI = PI.reshape(5, 5)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(PI, cmap='viridis', origin='lower')
ax.set_title('Persistence Image (5×5)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()
print(f'PI shape: {PI.shape}, sum: {PI.sum():.4f}')



## 3-way 비교 결과

8개 데이터셋에서 TLC-GNN exact PI / PDGNN approx PI / No PI 50-trial 평균.


In [ ]:

import glob
import re

def read_aucs(path):
    if not os.path.exists(path):
        return None
    aucs = []
    with open(path) as f:
        next(f)
        for line in f:
            parts = [p.strip() for p in line.split(',') if p.strip()]
            if len(parts) >= 3 and parts[0].isdigit():
                aucs.append(float(parts[2]))
    return aucs

datasets = ['Photo', 'PubMed', 'Computers', 'Chameleon', 'Squirrel',
            'Texas', 'Cornell', 'Wisconsin', 'ChChMiner']
tag_map = {
    'TLC-GNN': ['rerun', 'hetero', 'drug'],
    'PDGNN': ['pdgnn'],
    'No PI': ['heteroNoPI', 'drugNoPI'],
}

rows = []
for d in datasets:
    row = [d]
    for col_name, tags in tag_map.items():
        val = '-'
        for tag in tags:
            p = f'../scores/pipe_benchmark_{d}_LP_scores{tag}.txt'
            aucs = read_aucs(p)
            if aucs:
                val = f'{np.mean(aucs):.4f} ± {np.std(aucs):.4f} (n={len(aucs)})'
                break
        row.append(val)
    rows.append(row)

# Print as markdown table
print(f"| {'Dataset':<12} | {'TLC-GNN':<25} | {'PDGNN':<25} | {'No PI':<25} |")
print(f"| {'-'*12} | {'-'*25} | {'-'*25} | {'-'*25} |")
for r in rows:
    print(f"| {r[0]:<12} | {r[1]:<25} | {r[2]:<25} | {r[3]:<25} |")



## SBM Density × Heterophily Sweep

25 합성 그래프 (5×5 grid) × 3 variants 결과를 heatmap으로.


In [ ]:

from IPython.display import Image, display
heatmap_path = '../docs/figures/sbm_heatmap.png'
if os.path.exists(heatmap_path):
    display(Image(heatmap_path))
else:
    print(f'(heatmap not yet generated at {heatmap_path} — run Task 1.6)')



## Adaptive PI Gating

모델이 per-edge로 gate ∈ [0,1] 학습 → PI 사용량 자동 결정.


In [ ]:

gated_datasets = ['Photo', 'Chameleon', 'Texas', 'ChChMiner']
print(f"| {'Dataset':<12} | {'Gated AUC':<25} |")
print(f"| {'-'*12} | {'-'*25} |")
for d in gated_datasets:
    p = f'../scores/pipe_benchmark_{d}_LP_scoresgated.txt'
    aucs = read_aucs(p)
    if aucs:
        val = f'{np.mean(aucs):.4f} ± {np.std(aucs):.4f} (n={len(aucs)})'
    else:
        val = '(running)'
    print(f"| {d:<12} | {val:<25} |")



## 결론

1. **PI가 도움 = homophilic** (Photo / PubMed / Computers)
2. **PI가 무용/유해 = heterophilic + drug** (Chameleon / Squirrel / WebKB / ChChMiner)
3. **PDGNN approximation > exact PI** on homophilic (smoothing 효과)
4. **Adaptive gating**: 자동 의사결정 시도 (결과는 위 표)

GitHub: github.com/jjune5/TDA_conference
